# **Nebula Core** — train our own model (free T4)

Runtime → **Change runtime type** → **T4 GPU** → Run all.

This notebook trains **Nebula Core**, our own fine-tuned coding/build model,
on the Nebula corpus (website planning, production section code, design tokens,
copy, QA layout laws, GitHub flows, platform ops). Result: a GGUF model the
Nebula Worker can serve through the `custom` engine — **zero third-party API keys**.

3B model ≈ 60–90 min on a free T4. 7B ≈ 2.5–3 h.

In [ ]:
%pip install -q unsloth
# unsloth pulls torch/transformers/trl/datasets with pinned compatible versions

In [ ]:
# ── Config ──────────────────────────────────────────────────────
BASE       = "unsloth/Qwen2.5-Coder-3B-Instruct"  # or unsloth/Qwen2.5-Coder-7B-Instruct
EPOCHS     = 3
LR         = 2e-4
MAX_SEQ    = 4096
OUT_DIR    = "/content/nebula-core"
GGUF_DIR   = f"{OUT_DIR}/gguf"
print("config ready")

In [ ]:
# ── Dataset: fetch the committed Nebula corpus ─────────────────
# Option A (default): pull the latest corpus straight from the repo.
import urllib.request, pathlib
RAW = "https://raw.githubusercontent.com/bahyam-blip/nebula-crm/main/ml/dataset/nebula-core.jsonl"
pathlib.Path("/content").mkdir(exist_ok=True)
urllib.request.urlretrieve(RAW, "/content/nebula-core.jsonl")
# Option B: upload your own file to /content/nebula-core.jsonl.
n = sum(1 for _ in open("/content/nebula-core.jsonl"))
print(f"dataset ready: {n} rows")

In [ ]:
# ── Train (QLoRA, Unsloth) ─────────────────────────────────────
import json, os, torch
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE, max_seq_length=MAX_SEQ, dtype=None, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=32, lora_dropout=0.0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=3407,
)

rows = [{"messages": json.loads(l)["messages"]}
        for l in open("/content/nebula-core.jsonl") if l.strip()]
ds = Dataset.from_list(rows).map(
    lambda e: {"text": tokenizer.apply_chat_template(e["messages"], tokenize=False)},
    remove_columns=["messages"])
print(f"{len(ds)} rows")

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    dataset_text_field="text", max_seq_length=MAX_SEQ, packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        num_train_epochs=EPOCHS, warmup_ratio=0.03, learning_rate=LR,
        fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="cosine", seed=3407, output_dir=OUT_DIR,
        save_strategy="no", report_to="none",
    ),
)
trainer.train()

In [ ]:
# ── Quick sanity generation (before export) ────────────────────
FastLanguageModel.for_inference(model)
msgs = [
    {"role": "system", "content": "You are Nebula Core, the fine-tuned engineering model of the Nebula platform. You build production-quality websites on the FIRST round."},
    {"role": "user", "content": "Section: cta. Requirement: a closing CTA band: one promise, one button, no clutter. Business: dental clinic group. Brand tokens: bg #faf7f2, ink #191919, accent #0e7c66. Output the complete <section> with scoped <style>."},
]
inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
out = model.generate(input_ids=inputs, max_new_tokens=700, temperature=0.4)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)[:1200])

In [ ]:
# ── Export GGUF (q4_k_m + q5_k_m) ───────────────────────────────
os.makedirs(GGUF_DIR, exist_ok=True)
model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q4_k_m")
model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q5_k_m")
for f in os.listdir(GGUF_DIR):
    p = os.path.join(GGUF_DIR, f)
    if os.path.isfile(p):
        print(f"{f}  ({os.path.getsize(p)/1e9:.2f} GB)")

In [ ]:
# ── Persist GGUF to Google Drive (survives session end) ────────
from google.colab import drive
drive.mount('/content/drive')
import shutil, glob, os
dst_dir = '/content/drive/MyDrive/nebula-core'
os.makedirs(dst_dir, exist_ok=True)
for f in glob.glob(GGUF_DIR + '/*.gguf'):
    dst = os.path.join(dst_dir, os.path.basename(f))
    if not os.path.exists(dst):
        shutil.copy(f, dst)
        print('saved ->', dst)
print('Drive copy complete:', os.listdir(dst_dir))

In [ ]:
# ── Download the GGUF (or upload to HF) ────────────────────────
from google.colab import files
import glob
q4 = glob.glob(f"{GGUF_DIR}/*q4_k_m*.gguf")
if q4:
    files.download(q4[0])   # ~2 GB — takes a few minutes
else:
    print("no q4_k_m found — check previous cell")

# Optional: private HF hosting instead of local download
# !pip install -q huggingface_hub
# from huggingface_hub import login; login()
# from huggingface_hub import HfApi
# HfApi().upload_folder(folder_path=GGUF_DIR, repo_id="YOURUSER/nebula-core", repo_type="model", private=True)

## Serve it + wire it into Nebula (no third-party keys)

On any machine with 8 GB RAM+ (or a free-tier VM):
```bash
brew install llama.cpp   # or build from source / apt
llama-server -m nebula-core-q4_k_m.gguf --port 8080 --host 0.0.0.0
```
Then set the Worker vars (Cloudflare dashboard → Settings → Variables, or `wrangler secret put`):
```
LLM_ENGINE=custom
LLM_CUSTOM_BASE_URL=https://your-host:8080/v1
LLM_CUSTOM_MODEL=nebula-core
```
The router (`llm.js`) now prefers OUR model for every agent call —
Workers AI stays as the free fallback and Sarvam is never billed.
Verify: `GET /v1/ai/engine` → `{"active":"custom",...}`.